In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
# sample first character from the model

g = torch.Generator().manual_seed(2147483647)
p = N[0].float()
p = p / p.sum()

m_ixs = torch.multinomial(p, num_samples=5, replacement=True, generator=g)
ixs = [ix.item() for ix in m_ixs]       # [13, 19, 14, 1, 1]
chars_sample = [itos[ix] for ix in ixs] # ['m', 's', 'n', 'a', 'a']

In [ ]:
# sample words from the model (non-vectorized version)

g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    ix = 0 # we need first special token to start a word
    out = []    
    while True:
        p = N[ix].float()
        p = p / p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break

    print(''.join(out))


In [ ]:
# sample words from the model (vectorized version)

g = torch.Generator().manual_seed(2147483647)
P = N / N.float().sum(1, keepdim=True)

print(P[0].sum())
print(P.shape)

for i in range(10):
    ix = 0 # we need first special token to start a word
    out = []    
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break

    print(''.join(out))

In [ ]:
# evaluate the model
log_likelihood = 0.0
n = 0

# model smoothing, try other values also
P = (N+1).float()
P /= P.float().sum(1, keepdim=True)

for w in ['andrejq']:# words: # try evaluate a probability on any other word/name
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1 
    print(f'{ch1}{ch2}: {prob: .4f} {logprob: .4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n=}')
